# YOLOv8 Pruned Model Recovery

Finetune an already-pruned YOLOv8 checkpoint without rebuilding the original YOLOv8 architecture.


In [ ]:
%pip install -q ultralytics torch-pruning

In [ ]:
import os
import sys
import types
import random
import shutil
import zipfile
import urllib.request
import importlib
from pathlib import Path
from typing import Any

import torch

# Needed by some full-module pruned checkpoints during torch.load().
torch._utils = importlib.import_module("torch._utils")

from ultralytics import YOLO
from ultralytics.engine.model import checks, RANK, load_checkpoint

# Needed if the checkpoint contains the custom C2fV2 class.
from prune_yolo import C2fV2
import __main__
__main__.C2fV2 = C2fV2

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Workdir:", os.getcwd())
print("torch._utils:", hasattr(torch, "_utils"))

In [ ]:
RAW_PRUNED_WEIGHTS = "yolov8n_pruned_01.pt"
BASE_WEIGHTS = "yolov8n.pt"
DATA = "/content/datasets/coco2017_subset/coco2017_subset.yaml"

IMGSZ = 640
BATCH = 8
EPOCHS = 10
PATIENCE = 20
DEVICE = "0" if torch.cuda.is_available() else "cpu"
PROJECT = "runs/prune_recovery"

print("Pruned model:", RAW_PRUNED_WEIGHTS)
print("Dataset:", DATA)
print("Device:", DEVICE)
print()
print("Available .pt files:")
for p in sorted(Path(".").glob("*.pt")):
    print(f"  {p.name:45s} {p.stat().st_size / 1024 / 1024:6.2f} MB")

In [ ]:
def download_file(url: str, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        print("Already exists:", out_path)
        return
    print("Downloading:", url)
    urllib.request.urlretrieve(url, out_path)


def unzip_if_needed(zip_path: Path, expected_dir: Path, dst: Path):
    if expected_dir.exists():
        print("Already extracted:", expected_dir)
        return
    print("Unzipping:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dst)


def make_coco_val_subset(root="/content/datasets", n_train=2000, n_val=500, seed=0):
    root = Path(root)
    coco_root = root / "coco"
    subset_root = root / "coco2017_subset"
    random.seed(seed)

    labels_zip = root / "coco2017labels.zip"
    val_zip = coco_root / "images" / "val2017.zip"

    download_file(
        "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip",
        labels_zip,
    )
    download_file("http://images.cocodataset.org/zips/val2017.zip", val_zip)

    unzip_if_needed(labels_zip, coco_root / "labels" / "val2017", root)
    unzip_if_needed(val_zip, coco_root / "images" / "val2017", coco_root / "images")

    src_img = coco_root / "images" / "val2017"
    src_lbl = coco_root / "labels" / "val2017"
    images = sorted(src_img.glob("*.jpg"))
    if len(images) < n_train + n_val:
        raise ValueError(f"Need {n_train + n_val} images but found {len(images)}")

    # Only the generated subset has train/val folders. Clear old subset files first.
    for subdir in ["images/train", "images/val", "labels/train", "labels/val"]:
        split_dir = subset_root / subdir
        if split_dir.exists():
            shutil.rmtree(split_dir)
        split_dir.mkdir(parents=True, exist_ok=True)

    selected = random.sample(images, n_train + n_val)
    splits = {"train": selected[:n_train], "val": selected[n_train:]}

    for split, split_images in splits.items():
        for img_path in split_images:
            shutil.copy2(img_path, subset_root / f"images/{split}" / img_path.name)
            src_label = src_lbl / f"{img_path.stem}.txt"
            dst_label = subset_root / f"labels/{split}" / f"{img_path.stem}.txt"
            shutil.copy2(src_label, dst_label) if src_label.exists() else dst_label.touch()

    names = [
        "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck",
        "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench",
        "bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra",
        "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
        "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
        "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
        "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
        "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
        "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
        "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
        "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
        "hair drier", "toothbrush",
    ]

    yaml_path = subset_root / "coco2017_subset.yaml"
    yaml_path.write_text(
        f"path: {subset_root}\n"
        "train: images/train\n"
        "val: images/val\n\n"
        "nc: 80\n"
        f"names: {names}\n"
    )

    print("Dataset YAML:", yaml_path)
    print("Train images:", len(splits["train"]))
    print("Val images:", len(splits["val"]))
    return str(yaml_path)


DATA = make_coco_val_subset(n_train=2000, n_val=500)


In [ ]:
def load_pruned_yolo(weights: str, base_weights: str = BASE_WEIGHTS) -> YOLO:
    ckpt = torch.load(weights, map_location="cpu", weights_only=False)

    # Normal YOLO checkpoint fallback.
    if not (isinstance(ckpt, dict) and "model" in ckpt):
        return YOLO(weights)

    yolo = YOLO(base_weights)
    yolo.model = ckpt["model"].float()
    yolo.ckpt = ckpt
    yolo.task = getattr(yolo.model, "task", "detect")
    yolo.overrides = getattr(yolo.model, "args", {}) or {}
    yolo.ckpt_path = weights
    return yolo


def print_metrics(title: str, metrics):
    print()
    print(title)
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall:    {metrics.box.mr:.4f}")
    print(f"mAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")


def validate_model(model_or_weights, title="Metrics"):
    model = load_pruned_yolo(str(model_or_weights)) if isinstance(model_or_weights, (str, Path)) else model_or_weights
    metrics = model.val(data=DATA, imgsz=IMGSZ, device=DEVICE, batch=1)
    print_metrics(title, metrics)
    return metrics

In [ ]:
def train_pruned(self: YOLO, **kwargs: Any):
    self._check_is_pytorch_model()
    checks.check_pip_update_available()

    args = {
        **(getattr(self, "overrides", {}) or {}),
        **kwargs,
        "mode": "train",
        "task": self.task,
        "model": self.ckpt_path or "pruned_model.pt",
        "session": self.session,
    }

    self.trainer = self._smart_load("trainer")(overrides=args, _callbacks=self.callbacks)
    self.trainer.model = self.model
    self.model = self.trainer.model

    for p in self.model.parameters():
        if p.dtype.is_floating_point:
            p.requires_grad = True

    self.trainer.train()

    if RANK in {-1, 0}:
        best_or_last = self.trainer.best if self.trainer.best.exists() else self.trainer.last
        self.model, self.ckpt = load_checkpoint(best_or_last)
        self.overrides = self._reset_ckpt_args(self.model.args)
        self.metrics = getattr(self.trainer.validator, "metrics", None)

    return self.metrics


def finetune_pruned(weights: str):
    model = load_pruned_yolo(weights)
    model.train_pruned = types.MethodType(train_pruned, model)

    metrics = model.train_pruned(
        data=DATA,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=PROJECT,
        name=Path(weights).stem + "_finetune",
        exist_ok=True,
        pretrained=False,
        amp=False,
        patience=PATIENCE,
    )

    best_path = Path(model.trainer.best)
    last_path = Path(model.trainer.last)
    print("Best:", best_path)
    print("Last:", last_path)
    return model, metrics, best_path, last_path

In [ ]:
def run_recovery(weights: str = RAW_PRUNED_WEIGHTS):
    raw_metrics = validate_model(weights, "Raw-pruned metrics")

    model, train_metrics, best_path, last_path = finetune_pruned(weights)

    best_metrics = validate_model(best_path, "Recovered best metrics")

    return {
        "input": weights,
        "best": str(best_path),
        "last": str(last_path),
        "raw_map50": raw_metrics.box.map50,
        "raw_map": raw_metrics.box.map,
        "best_map50": best_metrics.box.map50,
        "best_map": best_metrics.box.map,
    }


result = run_recovery(RAW_PRUNED_WEIGHTS)
print(result)

In [ ]:
def prune_once(input_weights: str, ratio: float, output_weights: str):
    raise NotImplementedError("Add pruning code here next.")


def prune_finetune_loop(start_weights: str, ratios: list[float]):
    current = start_weights
    history = []

    for ratio in ratios:
        raw_out = f"{Path(current).stem}_pruned_{ratio:.3f}.pt"
        prune_once(current, ratio, raw_out)

        raw_metrics = validate_model(raw_out, f"Raw pruned {ratio}")
        _, _, best_path, _ = finetune_pruned(raw_out)
        best_metrics = validate_model(best_path, f"Recovered {ratio}")

        history.append({
            "ratio": ratio,
            "raw": raw_out,
            "best": str(best_path),
            "raw_map": raw_metrics.box.map,
            "best_map": best_metrics.box.map,
        })
        current = str(best_path)

    return history